# Reasoning

Reasoning model prompts tend to be high level and not as detailed as chat model prompts. Reasoning models work by generating a bunch of reasoning tokens that are not part of the user input or output. Setting the `reasoning_effort` controls the number of reasoning tokens that are generated. It can take three values - "high", "medium", and "low".

The documentation suggests using the new Responses API instead of the chat API for this because it gives two additional benefits -
  * Advanced function calling scenarios need the reasoning context to be carried over to subsequent calls to the LLM. Responses stateful API makes this possible. Chat API will crop off any reasoning context from the output so there is no way to carry it forward.
  * If I want a summary of the reasoning, then Responses API has a way to get it, Chat does not.

In [1]:
from dotenv import load_dotenv
from openai import OpenAI
from utils import LLM

In [2]:
load_dotenv()

True

In [3]:
client = OpenAI()

In [4]:
prompt = """
Write a bash script that takes a matrix represented as a string with
format '[1,2],[3,4],[5,6]' and prints the transpose in the same format.
"""

In [5]:
completion = client.chat.completions.create(
    model=LLM.RES_MINI,
    reasoning_effort="medium",
    messages=[{"role": "user", "content": prompt}],
)
completion

ChatCompletion(id='chatcmpl-BNBbKXJe38YXmWoMpeViyXJgYXKqM', choices=[Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content='Here’s a standalone Bash script, named for example transpose.sh, that takes a single argument in the form  \n"[1,2],[3,4],[5,6]"  \nand prints its transpose in the same format:\n\n```bash\n#!/usr/bin/env bash\n#\n# transpose.sh  — transpose a matrix given as "[1,2],[3,4],[5,6]"\n\nset -euo pipefail\n\n# read input from $1 or from stdin if no args\nif [ $# -ge 1 ]; then\n  input="$1"\nelse\n  read -r input\nfi\n\n# strip the outer [ ] and replace "],["\n# with spaces so we can split into rows\nclean=$(sed \'s/^\\[//; s/\\]$//; s/\\],\\[/ /g\' <<<"$input")\n\n# rows will be things like "1,2" "3,4" "5,6"\nread -ra rows <<<"$clean"\nm=${#rows[@]}          # number of rows\n\n# figure out number of columns by splitting the first row\nIFS=, read -ra tmp <<<"${rows[0]}"\nn=${#tmp[@]}           # number of columns\n\n# build the transp

```python
ChatCompletion(
    id='chatcmpl-BNBbKXJe38YXmWoMpeViyXJgYXKqM', 
    choices=[
        Choice(
            finish_reason='stop', 
            index=0, 
            logprobs=None, 
            message=ChatCompletionMessage(
                content='Here’s a standalone Bash script...', 
                refusal=None, 
                role='assistant', 
                annotations=[], 
                audio=None, 
                function_call=None, 
                tool_calls=None
            )
        )
    ], 
    created=1744865838, 
    model='o4-mini-2025-04-16', 
    object='chat.completion', 
    service_tier='default', 
    system_fingerprint=None, 
    usage=CompletionUsage(
        completion_tokens=1488, 
        prompt_tokens=44, 
        total_tokens=1532, 
        completion_tokens_details=CompletionTokensDetails(
            accepted_prediction_tokens=0, 
            audio_tokens=0, 
            reasoning_tokens=960, 
            rejected_prediction_tokens=0
        ), 
        prompt_tokens_details=PromptTokensDetails(audio_tokens=0, cached_tokens=0)
    )
)
```

In [6]:
print(completion.choices[0].message.content)

Here’s a standalone Bash script, named for example transpose.sh, that takes a single argument in the form  
"[1,2],[3,4],[5,6]"  
and prints its transpose in the same format:

```bash
#!/usr/bin/env bash
#
# transpose.sh  — transpose a matrix given as "[1,2],[3,4],[5,6]"

set -euo pipefail

# read input from $1 or from stdin if no args
if [ $# -ge 1 ]; then
  input="$1"
else
  read -r input
fi

# strip the outer [ ] and replace "],["
# with spaces so we can split into rows
clean=$(sed 's/^\[//; s/\]$//; s/\],\[/ /g' <<<"$input")

# rows will be things like "1,2" "3,4" "5,6"
read -ra rows <<<"$clean"
m=${#rows[@]}          # number of rows

# figure out number of columns by splitting the first row
IFS=, read -ra tmp <<<"${rows[0]}"
n=${#tmp[@]}           # number of columns

# build the transpose
out=()
for ((j=0; j<n; j++)); do
  rowj=()
  for ((i=0; i<m; i++)); do
    IFS=, read -ra elems <<<"${rows[i]}"
    rowj+=("${elems[j]}")
  done
  # join rowj by comma and wrap in [ ]
  IFS=,; 

Here are some more examples from the documentation.

In [8]:
prompt = """
Instructions:
  * Given the React component below, change it so that nonfiction books have red text.
  * Return only the code in your reply.
  * Do not include any additional formatting, such as markdown code blocks.
  * For formatting, use four space tabs, and do not allow any lines of code to exceed 80 columns.

const books = [
    {title: 'Dune', category: 'fiction', id: 1},
    {title: 'Frankenstien', category: 'fiction', id: 2},
    {title: 'Moneyball', category: 'nonfiction', id: 3}
];

export default function BookList() {
    const listItems = books.map(book => 
        <li>
            {book.title}
        </li>
    );

    return (
        <ul>
            {listItems}
        </ul>
    );
}
"""

message = {"role": "user", "content": [{"type": "text", "text": prompt}]}

In [9]:
completion = client.chat.completions.create(
    model=LLM.RES_MINI, messages=[message]  # type: ignore
)
completion

ChatCompletion(id='chatcmpl-BND02ObIIvYg5fw1jvcK9SHLPaOfB', choices=[Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content="const books = [\n    {title: 'Dune', category: 'fiction', id: 1},\n    {title: 'Frankenstien', category: 'fiction', id: 2},\n    {title: 'Moneyball', category: 'nonfiction', id: 3}\n];\n\nexport default function BookList() {\n    const listItems = books.map(book =>\n        <li\n            key={book.id}\n            style={book.category === 'nonfiction'\n                ? {color: 'red'}\n                : {}}\n        >\n            {book.title}\n        </li>\n    );\n\n    return (\n        <ul>\n            {listItems}\n        </ul>\n    );\n}", refusal=None, role='assistant', annotations=[], audio=None, function_call=None, tool_calls=None))], created=1744871214, model='o4-mini-2025-04-16', object='chat.completion', service_tier='default', system_fingerprint=None, usage=CompletionUsage(completion_tokens=2017, prompt_tokens

```python
ChatCompletion(
    id='chatcmpl-BND02ObIIvYg5fw1jvcK9SHLPaOfB', 
    choices=[
        Choice(
            finish_reason='stop', 
            index=0, 
            logprobs=None, 
            message=ChatCompletionMessage(
                content="const books = ...", 
                refusal=None, 
                role='assistant', 
                annotations=[], 
                audio=None, 
                function_call=None, 
                tool_calls=None
            )
        )
    ], 
    created=1744871214, 
    model='o4-mini-2025-04-16', 
    object='chat.completion', 
    service_tier='default', 
    system_fingerprint=None, 
    usage=CompletionUsage(
        completion_tokens=2017, 
        prompt_tokens=189, 
        total_tokens=2206, 
        completion_tokens_details=CompletionTokensDetails(
            accepted_prediction_tokens=0, 
            audio_tokens=0, 
            reasoning_tokens=1856, 
            rejected_prediction_tokens=0
        ), 
        prompt_tokens_details=PromptTokensDetails(audio_tokens=0, cached_tokens=0)
    )
)
```

In [11]:
print(completion.choices[0].message.content)

const books = [
    {title: 'Dune', category: 'fiction', id: 1},
    {title: 'Frankenstien', category: 'fiction', id: 2},
    {title: 'Moneyball', category: 'nonfiction', id: 3}
];

export default function BookList() {
    const listItems = books.map(book =>
        <li
            key={book.id}
            style={book.category === 'nonfiction'
                ? {color: 'red'}
                : {}}
        >
            {book.title}
        </li>
    );

    return (
        <ul>
            {listItems}
        </ul>
    );
}


In [12]:
prompt = """
I want to build a Python app that takes user questions and looks them up in a database where they are mapped to answers. If there is
a close match, it retrieves the matched answer. If there isn't, it asks the user to provide an answer and stores the question/answer
pair in the database. Make a plan for the directory structure you'll need, then return each file in full. Only supply your reasoning
at the beginning and end, not throughout the code.
"""
message = {"role": "user", "content": [{"type": "text", "text": prompt}]}

In [13]:
completion = client.chat.completions.create(
    model=LLM.RES_MINI, messages=[message]  # type: ignore
)
completion

ChatCompletion(id='chatcmpl-BND5eqkJUZlF6AiNqVObmttRlrpfG', choices=[Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content='Below is a simple CLI‑based Python application that:\n\n1. Stores Q&A pairs in an SQLite database.  \n2. On each user query, looks for a close match (using difflib).  \n3. If found, returns the stored answer; otherwise asks the user for a new answer and saves it.\n\nDirectory structure:\n\nmy_qa_app/\n│\n├── .gitignore\n├── README.md\n├── requirements.txt\n└── app\n    ├── __init__.py\n    ├── db.py\n    ├── utils.py\n    └── main.py\n\nFile contents follow in full.\n\n––––––––––––––––––––––––––––––––––––––––––––––––––––––––––––––––––\nFile: my_qa_app/.gitignore\n––––––––––––––––––––––––––––––––––––––––––––––––––––––––––––––––––\n__pycache__/\n*.db\n\n––––––––––––––––––––––––––––––––––––––––––––––––––––––––––––––––––\nFile: my_qa_app/requirements.txt\n––––––––––––––––––––––––––––––––––––––––––––––––––––––––––––––––––\n# No exte

```python
ChatCompletion(
    id='chatcmpl-BND5eqkJUZlF6AiNqVObmttRlrpfG', 
    choices=[
        Choice(
            finish_reason='stop', 
            index=0, 
            logprobs=None, 
            message=ChatCompletionMessage(
                content='....', 
                refusal=None, 
                role='assistant', 
                annotations=[], 
                audio=None, 
                function_call=None, 
                tool_calls=None
            )
        )
    ], 
    created=1744871562, 
    model='o4-mini-2025-04-16', 
    object='chat.completion', 
    service_tier='default', 
    system_fingerprint=None, 
    usage=CompletionUsage(
        completion_tokens=2388, 
        prompt_tokens=103, 
        total_tokens=2491, 
        completion_tokens_details=CompletionTokensDetails(
            accepted_prediction_tokens=0, 
            audio_tokens=0, 
            reasoning_tokens=1152, 
            rejected_prediction_tokens=0
        ), 
        prompt_tokens_details=PromptTokensDetails(audio_tokens=0, cached_tokens=0)
    )
)
```

In [15]:
print(completion.choices[0].message.content)

Below is a simple CLI‑based Python application that:

1. Stores Q&A pairs in an SQLite database.  
2. On each user query, looks for a close match (using difflib).  
3. If found, returns the stored answer; otherwise asks the user for a new answer and saves it.

Directory structure:

my_qa_app/
│
├── .gitignore
├── README.md
├── requirements.txt
└── app
    ├── __init__.py
    ├── db.py
    ├── utils.py
    └── main.py

File contents follow in full.

––––––––––––––––––––––––––––––––––––––––––––––––––––––––––––––––––
File: my_qa_app/.gitignore
––––––––––––––––––––––––––––––––––––––––––––––––––––––––––––––––––
__pycache__/
*.db

––––––––––––––––––––––––––––––––––––––––––––––––––––––––––––––––––
File: my_qa_app/requirements.txt
––––––––––––––––––––––––––––––––––––––––––––––––––––––––––––––––––
# No external dependencies  
# Uses only Python standard library (sqlite3, difflib)

––––––––––––––––––––––––––––––––––––––––––––––––––––––––––––––––––
File: my_qa_app/README.md
––––––––––––––––––––––